# EV Advanced Aggregation, Pivot Tables, Cross-Tabulation & Feature Engineering

## Professional Analytical Transformation Notebook

### Project
**EV Intelligence Platform — Electric Vehicle Population Data**

---

## 🎯 Notebook Objective

This notebook takes the cleaned EV dataset from **Notebook 01** and the exploratory understanding from **Notebook 02** and moves into the next analytical layer:

```text
CLEAN DATA
    ↓
GROUPBY
    ↓
AGGREGATION
    ↓
PIVOT TABLES
    ↓
CROSSTABULATION
    ↓
MULTI-DIMENSIONAL SUMMARIES
    ↓
RANKING
    ↓
PERCENTAGE / SHARE ANALYSIS
    ↓
COMPARATIVE ANALYSIS
    ↓
FEATURE ENGINEERING
    ↓
ANALYTICAL DATASET
```

The purpose is not merely to calculate counts. The goal is to transform row-level vehicle records into **business-ready analytical tables and meaningful derived features**.

---

# 🔎 Dataset Columns Used

Expected cleaned columns:

- `VIN`
- `County`
- `City`
- `State`
- `Postal_Code`
- `Model_Year`
- `Make`
- `Model`
- `EV_Type`
- `CAFV_Eligibility`
- `Electric_Range`
- `Base_MSRP`
- `Legislative District`
- `Vehicle_ID`
- `Vehicle_Location`
- `Electric_Utility`
- `Census_Tract`

### Important analytical rule

This dataset represents **vehicle population records**. Therefore, a row count should be interpreted as a vehicle-record count, not automatically as sales, revenue, profit, or market share.

---

# 🧠 What We Will Learn

### GroupBy
Use `groupby()` to answer questions such as:

- How many vehicles does each manufacturer have?
- What is the average range by EV type?
- What is the median MSRP by manufacturer?
- Which states have the most vehicles?
- How does model-year composition vary?

### Aggregation
Use:

- `count`
- `size`
- `nunique`
- `sum`
- `mean`
- `median`
- `min`
- `max`
- `std`
- `var`
- `quantile`

### Pivot Tables

Create spreadsheet-style analytical summaries with:

- rows
- columns
- values
- aggregation functions
- margins
- percentages

### Cross Tabulation

Analyze relationships between categorical variables:

```text
State × EV Type
Make × EV Type
Make × CAFV Eligibility
State × CAFV Eligibility
Model Year × EV Type
```

### Feature Engineering

Create analytical features such as:

- Vehicle Age
- Vehicle Age Group
- Range Group
- MSRP Group
- Model Year Group
- Range per MSRP
- MSRP per Range
- Is Zero Range
- Is Zero MSRP
- Manufacturer Rank
- State Rank
- Make Share
- State Share
- EV Type Share
- CAFV Share
- Range Z-score
- MSRP Z-score
- Geographic and vehicle-level indicators

---

# ⚠️ Feature Engineering Principle

Derived features must be based on fields that actually exist.

Do **not** invent:

- revenue
- profit
- customer income
- battery capacity
- charging speed
- annual mileage
- sales volume

unless those fields are genuinely available in the source data.



## 1. Environment & Dataset Setup

In [1]:
# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("✅ Libraries imported.")

✅ Libraries imported.


In [2]:
# 2. Load Cleaned & Renamed Dataset

from pathlib import Path

file_path = Path("../datas/cleaned/ev_cleaned.csv")

if not file_path.exists():
    raise FileNotFoundError(
        f"Cleaned dataset not found: {file_path.resolve()}"
    )

df = pd.read_csv(file_path)

print("✅ Cleaned & renamed dataset loaded!")
print("Shape:", df.shape)

display(df.head())

✅ Cleaned & renamed dataset loaded!
Shape: (150482, 17)


C:\Users\Admin\AppData\Local\Temp\ipykernel_2892\320378072.py:12: DtypeWarning: Columns (0: Postal_Code, 1: Census_Tract) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


,VIN,County,City,State,Postal_Code,Model_Year,Make,Model,EV_Type,CAFV_Eligibility,Electric_Range,Base_MSRP,Vehicle_ID,Vehicle_Location,Electric_Utility,Census_Tract,Legislative_District
0,KM8K33AGXL,King,Seattle,WA,"98,103.00",2020,HYUNDAI,KONA,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,258,0,249675142,POINT (-122.34301 47.659185),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),"53,033,004,800.00",43
1,1C4RJYB61N,King,Bothell,WA,"98,011.00",2022,JEEP,GRAND CHEROKEE,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,25,0,233928502,POINT (-122.20578 47.762405),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),"53,033,021,804.00",1
2,1C4RJYD61P,Yakima,Yakima,WA,"98,908.00",2023,JEEP,GRAND CHEROKEE,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,25,0,229675939,POINT (-120.6027202 46.5965625),PACIFICORP,"53,077,002,900.00",14
3,5YJ3E1EA7J,King,Kirkland,WA,"98,034.00",2018,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,215,0,104714466,POINT (-122.209285 47.71124),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),"53,033,021,903.00",45
4,WBY7Z8C5XJ,Thurston,Olympia,WA,"98,501.00",2018,BMW,I3,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,97,0,185498386,POINT (-122.89692 47.043535),PUGET SOUND ENERGY INC,"53,067,010,700.00",22


In [3]:
# 3. Confirm required columns
required_cols = [
    "VIN", "County", "City", "State", "Postal_Code", "Model_Year",
    "Make", "Model", "EV_Type", "CAFV_Eligibility", "Electric_Range",
    "Base_MSRP", "Vehicle_ID", "Vehicle_Location",
    "Electric_Utility", "Census_Tract"
]

available = [c for c in required_cols if c in df.columns]
missing = [c for c in required_cols if c not in df.columns]

print("Available required columns:", len(available))
print("Missing required columns:", missing)

Available required columns: 16
Missing required columns: []


In [4]:
# 4. Create a safe numeric column list
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

Numeric: ['Model_Year', 'Electric_Range', 'Base_MSRP', 'Vehicle_ID', 'Legislative_District']
Categorical: ['VIN', 'County', 'City', 'State', 'Postal_Code', 'Make', 'Model', 'EV_Type', 'CAFV_Eligibility', 'Vehicle_Location', 'Electric_Utility', 'Census_Tract']


## 2. GroupBy Fundamentals

In [5]:
# 5. Basic groupby count by Make
make_count = df.groupby("Make").size().sort_values(ascending=False)
display(make_count.to_frame("vehicle_records").head(30))

,vehicle_records
Make,
TESLA,68983
NISSAN,13497
CHEVROLET,12026
FORD,7614
BMW,6439
KIA,6198
TOYOTA,5223
VOLKSWAGEN,4074
VOLVO,3536


In [6]:
# 6. Groupby count with reset_index
make_count_df = (
    df.groupby("Make")
      .size()
      .reset_index(name="vehicle_records")
      .sort_values("vehicle_records", ascending=False)
)

display(make_count_df.head(30))

,Make,vehicle_records
31,TESLA,68983
25,NISSAN,13497
6,CHEVROLET,12026
10,FORD,7614
4,BMW,6439
16,KIA,6198
33,TOYOTA,5223
34,VOLKSWAGEN,4074
35,VOLVO,3536
15,JEEP,3292


In [7]:
df.columns

Index(['VIN', 'County', 'City', 'State', 'Postal_Code', 'Model_Year', 'Make',
       'Model', 'EV_Type', 'CAFV_Eligibility', 'Electric_Range', 'Base_MSRP',
       'Vehicle_ID', 'Vehicle_Location', 'Electric_Utility', 'Census_Tract',
       'Legislative_District'],
      dtype='str')

In [8]:
# 7. Groupby size vs count
comparison = pd.DataFrame({
    "size": df.groupby("Make").size(),
    "VIN_count": df.groupby("Make")["VIN"].count(),
    "unique_VIN": df.groupby("Make")["VIN"].nunique()
}).sort_values("size", ascending=False)

display(comparison.head(30))

,size,VIN_count,unique_VIN
Make,,,
TESLA,68983,68983,1303
NISSAN,13497,13497,365
CHEVROLET,12026,12026,738
FORD,7614,7614,814
BMW,6439,6439,724
KIA,6198,6198,522
TOYOTA,5223,5223,290
VOLKSWAGEN,4074,4074,327
VOLVO,3536,3536,943


In [9]:
# 8. GroupBy by EV type
ev_type_count = (
    df.groupby("EV_Type")
      .size()
      .reset_index(name="vehicle_records")
      .sort_values("vehicle_records", ascending=False)
)

display(ev_type_count)

,EV_Type,vehicle_records
0,Battery Electric Vehicle (BEV),116807
1,Plug-in Hybrid Electric Vehicle (PHEV),33675


In [10]:
# 9. GroupBy by State
state_count = (
    df.groupby("State")
      .size()
      .reset_index(name="vehicle_records")
      .sort_values("vehicle_records", ascending=False)
)

display(state_count.head(30))

,State,vehicle_records
39,WA,150141
6,CA,92
38,VA,35
21,MD,33
36,TX,20
26,NC,13
15,IL,12
7,CO,11
4,AZ,11
11,FL,9


In [11]:
# 10. GroupBy by Model Year
year_count = (
    df.groupby("Model_Year")
      .size()
      .reset_index(name="vehicle_records")
      .sort_values("Model_Year")
)

display(year_count.tail(25))

,Model_Year,vehicle_records
0,1997,1
1,1998,1
2,1999,4
3,2000,8
4,2002,2
5,2003,1
6,2008,19
7,2010,24
8,2011,796
9,2012,1633


## 3. Single-Variable Aggregation

In [12]:
# 11. Mean range by EV type
display(
    df.groupby("EV_Type")["Electric_Range"]
      .mean()
      .sort_values(ascending=False)
      .to_frame("average_range")
)

,average_range
EV_Type,
Battery Electric Vehicle (BEV),78.61
Plug-in Hybrid Electric Vehicle (PHEV),30.66


In [13]:
# 12. Median range by EV type
display(
    df.groupby("EV_Type")["Electric_Range"]
      .median()
      .sort_values(ascending=False)
      .to_frame("median_range")
)

,median_range
EV_Type,
Plug-in Hybrid Electric Vehicle (PHEV),26.00
Battery Electric Vehicle (BEV),0.00


In [14]:
# 13. Minimum and maximum range by EV type
display(
    df.groupby("EV_Type")["Electric_Range"]
      .agg(["min", "max"])
      .sort_values("max", ascending=False)
)

,min,max
EV_Type,,
Battery Electric Vehicle (BEV),0,337
Plug-in Hybrid Electric Vehicle (PHEV),6,153


In [15]:
# 14. Range standard deviation by EV type
display(
    df.groupby("EV_Type")["Electric_Range"]
      .std()
      .sort_values(ascending=False)
      .to_frame("range_std")
)

,range_std
EV_Type,
Battery Electric Vehicle (BEV),106.48
Plug-in Hybrid Electric Vehicle (PHEV),16.46


In [16]:
# 15. Average MSRP by EV type
display(
    df.groupby("EV_Type")["Base_MSRP"]
      .mean()
      .sort_values(ascending=False)
      .to_frame("average_msrp")
)

,average_msrp
EV_Type,
Plug-in Hybrid Electric Vehicle (PHEV),"1,889.88"
Battery Electric Vehicle (BEV),"1,146.23"


In [17]:
# 16. Median MSRP by EV type
display(
    df.groupby("EV_Type")["Base_MSRP"]
      .median()
      .sort_values(ascending=False)
      .to_frame("median_msrp")
)

,median_msrp
EV_Type,
Battery Electric Vehicle (BEV),0.00
Plug-in Hybrid Electric Vehicle (PHEV),0.00


## 4. Multi-Aggregation Analysis

In [18]:
# 17. Full range summary by EV type
range_summary = (
    df.groupby("EV_Type")["Electric_Range"]
      .agg(
          records="count",
          mean="mean",
          median="median",
          minimum="min",
          maximum="max",
          std="std"
      )
      .sort_values("records", ascending=False)
)

display(range_summary)

,records,mean,median,minimum,maximum,std
EV_Type,,,,,,
Battery Electric Vehicle (BEV),116807,78.61,0.00,0,337,106.48
Plug-in Hybrid Electric Vehicle (PHEV),33675,30.66,26.00,6,153,16.46


In [19]:
# 18. Full MSRP summary by EV type
msrp_summary = (
    df.groupby("EV_Type")["Base_MSRP"]
      .agg(
          records="count",
          mean="mean",
          median="median",
          minimum="min",
          maximum="max",
          std="std"
      )
      .sort_values("records", ascending=False)
)

display(msrp_summary)

,records,mean,median,minimum,maximum,std
EV_Type,,,,,,
Battery Electric Vehicle (BEV),116807,"1,146.23",0.00,0,110950,"8,553.83"
Plug-in Hybrid Electric Vehicle (PHEV),33675,"1,889.88",0.00,0,845000,"11,251.02"


In [20]:
# 19. Multiple numeric columns with multiple aggregations
multi_summary = (
    df.groupby("EV_Type")
      .agg({
          "Electric_Range": ["count", "mean", "median", "min", "max", "std"],
          "Base_MSRP": ["count", "mean", "median", "min", "max", "std"]
      })
)

display(multi_summary)

Electric_Range                        \
                                                count  mean median min  max   
EV_Type                                                                       
Battery Electric Vehicle (BEV)                 116807 78.61   0.00   0  337   
Plug-in Hybrid Electric Vehicle (PHEV)          33675 30.66  26.00   6  153   

                                              Base_MSRP                      \
                                          std     count     mean median min   
EV_Type                                                                       
Battery Electric Vehicle (BEV)         106.48    116807 1,146.23   0.00   0   
Plug-in Hybrid Electric Vehicle (PHEV)  16.46     33675 1,889.88   0.00   0   

                                                          
                                           max       std  
EV_Type                                                   
Battery Electric Vehicle (BEV)          110950  8,553.83  
Plug-in Hybrid Electric Vehicle (PHEV)  845000 11,251.02

In [21]:
# 20. Named aggregation — clean professional output
named_summary = (
    df.groupby("EV_Type")
      .agg(
          vehicle_records=("VIN", "count"),
          unique_models=("Model", "nunique"),
          unique_makes=("Make", "nunique"),
          avg_range=("Electric_Range", "mean"),
          median_range=("Electric_Range", "median"),
          max_range=("Electric_Range", "max"),
          avg_msrp=("Base_MSRP", "mean"),
          median_msrp=("Base_MSRP", "median")
      )
      .sort_values("vehicle_records", ascending=False)
)

display(named_summary.round(2))

,vehicle_records,unique_models,unique_makes,avg_range,median_range,max_range,avg_msrp,median_msrp
EV_Type,,,,,,,,
Battery Electric Vehicle (BEV),116807,67,27,78.61,0.00,337,"1,146.23",0.00
Plug-in Hybrid Electric Vehicle (PHEV),33675,63,25,30.66,26.00,153,"1,889.88",0.00


## 5. Manufacturer-Level Deep Aggregation

In [22]:
# 21. Manufacturer profile
make_profile = (
    df.groupby("Make")
      .agg(
          vehicle_records=("VIN", "count"),
          unique_models=("Model", "nunique"),
          avg_range=("Electric_Range", "mean"),
          median_range=("Electric_Range", "median"),
          max_range=("Electric_Range", "max"),
          avg_msrp=("Base_MSRP", "mean"),
          median_msrp=("Base_MSRP", "median"),
          ev_types=("EV_Type", "nunique")
      )
      .sort_values("vehicle_records", ascending=False)
)

display(make_profile.head(30).round(2))

,vehicle_records,unique_models,avg_range,median_range,max_range,avg_msrp,median_msrp,ev_types
Make,,,,,,,,
TESLA,68983,5,90.30,0.00,337,"1,648.60",0.00,1
NISSAN,13497,2,84.72,84.00,215,0.00,0.00,1
CHEVROLET,12026,5,95.27,53.00,259,0.00,0.00,2
FORD,7614,8,13.02,19.00,100,0.00,0.00,2
BMW,6439,11,38.16,30.00,153,"4,154.12",0.00,2
KIA,6198,7,52.62,26.00,239,"3,253.04",0.00,2
TOYOTA,5223,6,26.92,25.00,103,0.00,0.00,2
VOLKSWAGEN,4074,2,28.33,0.00,125,0.00,0.00,1
VOLVO,3536,7,15.63,18.00,41,"4,785.93",0.00,2


In [23]:
# 22. Manufacturer model diversity
model_diversity = (
    df.groupby("Make")["Model"]
      .nunique()
      .sort_values(ascending=False)
      .to_frame("unique_models")
)

display(model_diversity.head(30))

,unique_models
Make,
AUDI,11
BMW,11
MERCEDES-BENZ,10
FORD,8
HYUNDAI,8
VOLVO,7
KIA,7
TOYOTA,6
TESLA,5


In [24]:
# 23. Manufacturer range performance
make_range = (
    df.groupby("Make")["Electric_Range"]
      .agg(["count", "mean", "median", "max", "std"])
      .sort_values("median", ascending=False)
)

display(make_range.head(30).round(2))

,count,mean,median,max,std
Make,,,,,
JAGUAR,228,204.24,234.00,234,78.14
WHEEGO ELECTRIC CARS,3,100.00,100.00,100,0.00
TH!NK,5,100.00,100.00,100,0.00
FIAT,806,85.63,87.00,87,1.50
NISSAN,13497,84.72,84.00,215,51.41
SMART,274,62.38,58.00,68,4.97
AZURE DYNAMICS,9,56.00,56.00,56,0.00
CHEVROLET,12026,95.27,53.00,259,101.25
HONDA,810,46.63,47.00,48,3.57


In [25]:
# 24. Manufacturer MSRP profile
make_msrp = (
    df.groupby("Make")["Base_MSRP"]
      .agg(["count", "mean", "median", "min", "max", "std"])
      .sort_values("median", ascending=False)
)

display(make_msrp.head(30).round(2))

,count,mean,median,min,max,std
Make,,,,,,
FISKER,17,"102,000.00","102,000.00",102000,102000,0.00
WHEEGO ELECTRIC CARS,3,"32,995.00","32,995.00",32995,32995,0.00
AZURE DYNAMICS,9,0.00,0.00,0,0,0.00
AUDI,3006,0.00,0.00,0,0,0.00
ALFA ROMEO,12,0.00,0.00,0,0,0.00
BMW,6439,"4,154.12",0.00,0,91250,"14,529.12"
BENTLEY,2,0.00,0.00,0,0,0.00
CHRYSLER,2642,"1,771.16",0.00,0,39995,"8,229.60"
CADILLAC,176,"6,400.14",0.00,0,75095,"21,027.82"


In [26]:
# 25. Manufacturer and EV-type diversity
make_evtype = (
    df.groupby("Make")["EV_Type"]
      .nunique()
      .sort_values(ascending=False)
      .to_frame("unique_ev_types")
)

display(make_evtype.head(30))

,unique_ev_types
Make,
AUDI,2
BMW,2
FORD,2
CHEVROLET,2
CADILLAC,2
PORSCHE,2
SUBARU,2
VOLVO,2
HYUNDAI,2


## 6. Geographic Aggregation

In [27]:
# 26. State profile
state_profile = (
    df.groupby("State")
      .agg(
          vehicle_records=("VIN", "count"),
          unique_makes=("Make", "nunique"),
          unique_models=("Model", "nunique"),
          avg_range=("Electric_Range", "mean"),
          median_range=("Electric_Range", "median"),
          avg_msrp=("Base_MSRP", "mean"),
          median_msrp=("Base_MSRP", "median"),
          counties=("County", "nunique"),
          cities=("City", "nunique")
      )
      .sort_values("vehicle_records", ascending=False)
)

display(state_profile.head(30).round(2))

,vehicle_records,unique_makes,unique_models,avg_range,median_range,avg_msrp,median_msrp,counties,cities
State,,,,,,,,,
WA,150141,37,127,67.86,18.00,"1,311.28",0.00,39,461
CA,92,16,28,69.20,22.00,"1,998.91",0.00,21,51
VA,35,10,18,73.11,32.00,0.00,0.00,14,18
MD,33,14,20,57.36,30.00,"3,090.91",0.00,11,27
TX,20,8,12,80.35,23.00,"1,749.75",0.00,13,14
NC,13,4,6,40.69,0.00,0.00,0.00,8,9
IL,12,6,7,94.25,30.50,"3,332.92",0.00,7,11
CO,11,6,10,90.73,32.00,0.00,0.00,6,6
AZ,11,8,10,36.82,22.00,0.00,0.00,2,8


In [28]:
# 27. County profile
county_profile = (
    df.groupby("County")
      .agg(
          vehicle_records=("VIN", "count"),
          unique_makes=("Make", "nunique"),
          unique_models=("Model", "nunique"),
          avg_range=("Electric_Range", "mean"),
          avg_msrp=("Base_MSRP", "mean")
      )
      .sort_values("vehicle_records", ascending=False)
)

display(county_profile.head(30).round(2))

,vehicle_records,unique_makes,unique_models,avg_range,avg_msrp
County,,,,,
King,79078,35,123,68.57,"1,324.39"
Snohomish,17307,34,117,63.13,"1,155.08"
Pierce,11542,33,116,64.38,"1,463.42"
Clark,8855,33,110,65.59,"1,197.41"
Thurston,5403,32,100,69.34,994.67
Kitsap,4923,33,104,68.94,"1,609.18"
Spokane,3690,31,102,63.92,"1,175.37"
Whatcom,3668,34,103,71.41,"1,546.39"
Benton,1801,28,86,70.70,"1,214.81"


In [29]:
# 28. City profile
city_profile = (
    df.groupby("City")
      .agg(
          vehicle_records=("VIN", "count"),
          unique_makes=("Make", "nunique"),
          unique_models=("Model", "nunique"),
          avg_range=("Electric_Range", "mean"),
          avg_msrp=("Base_MSRP", "mean")
      )
      .sort_values("vehicle_records", ascending=False)
)

display(city_profile.head(30).round(2))

,vehicle_records,unique_makes,unique_models,avg_range,avg_msrp
City,,,,,
Seattle,25678,34,118,71.28,"1,234.78"
Bellevue,7691,32,108,72.08,"1,550.87"
Redmond,5502,32,100,70.72,"1,224.09"
Vancouver,5310,31,101,64.25,"1,171.61"
Bothell,4861,31,99,61.19,696.25
Kirkland,4622,31,103,76.48,"1,574.33"
Sammamish,4436,32,99,72.61,"1,473.15"
Renton,4043,30,97,60.17,"1,085.25"
Olympia,3634,31,93,70.79,"1,015.53"


In [30]:
# 29. Electric utility profile
if "Electric_Utility" in df.columns:
    utility_profile = (
        df.groupby("Electric_Utility")
          .agg(
              vehicle_records=("VIN", "count"),
              unique_makes=("Make", "nunique"),
              unique_models=("Model", "nunique")
          )
          .sort_values("vehicle_records", ascending=False)
    )
    display(utility_profile.head(30))

,vehicle_records,unique_makes,unique_models
Electric_Utility,,,
PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),55637,35,122
PUGET SOUND ENERGY INC,29865,36,121
CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),27268,34,118
BONNEVILLE POWER ADMINISTRATION||PUD NO 1 OF CLARK COUNTY - (WA),8643,33,109
BONNEVILLE POWER ADMINISTRATION||CITY OF TACOMA - (WA)||PENINSULA LIGHT COMPANY,6622,33,111
PUGET SOUND ENERGY INC||PUD NO 1 OF WHATCOM COUNTY,3443,33,102
BONNEVILLE POWER ADMINISTRATION||AVISTA CORP||INLAND POWER & LIGHT COMPANY,2244,30,93
BONNEVILLE POWER ADMINISTRATION||PUD 1 OF SNOHOMISH COUNTY,1341,30,84
PACIFICORP,1102,31,86


## 7. Time / Model-Year Aggregation

In [31]:
# 30. Model-year profile
year_profile = (
    df.groupby("Model_Year")
      .agg(
          vehicle_records=("VIN", "count"),
          unique_makes=("Make", "nunique"),
          unique_models=("Model", "nunique"),
          avg_range=("Electric_Range", "mean"),
          median_range=("Electric_Range", "median"),
          avg_msrp=("Base_MSRP", "mean"),
          median_msrp=("Base_MSRP", "median")
      )
      .sort_index()
)

display(year_profile.tail(30).round(2))

,vehicle_records,unique_makes,unique_models,avg_range,median_range,avg_msrp,median_msrp
Model_Year,,,,,,,
1997,1,1,1,39.00,39.00,0.00,0.00
1998,1,1,1,58.00,58.00,0.00,0.00
1999,4,1,1,74.00,74.00,0.00,0.00
2000,8,1,1,58.00,58.00,0.00,0.00
2002,2,1,1,95.00,95.00,0.00,0.00
2003,1,1,1,95.00,95.00,0.00,0.00
2008,19,1,1,220.00,220.00,"98,950.00","98,950.00"
2010,24,2,2,226.88,245.00,"101,205.62","110,950.00"
2011,796,5,5,70.85,73.00,958.54,0.00


In [32]:
# 31. Year-over-year change in record count
year_trend = year_profile[["vehicle_records"]].copy()
year_trend["yoy_change"] = year_trend["vehicle_records"].diff()
year_trend["yoy_pct"] = year_trend["vehicle_records"].pct_change().mul(100)

display(year_trend.tail(25).round(2))

,vehicle_records,yoy_change,yoy_pct
Model_Year,,,
1997,1,NaN,NaN
1998,1,0.00,0.00
1999,4,3.00,300.00
2000,8,4.00,100.00
2002,2,-6.00,-75.00
2003,1,-1.00,-50.00
2008,19,18.00,"1,800.00"
2010,24,5.00,26.32
2011,796,772.00,"3,216.67"


In [33]:
# 32. Manufacturer × model year
make_year = (
    df.groupby(["Model_Year", "Make"])
      .size()
      .reset_index(name="vehicle_records")
      .sort_values(["Model_Year", "vehicle_records"], ascending=[False, False])
)

display(make_year.head(40))

,Model_Year,Make,vehicle_records
223,2024,BMW,204
228,2024,MAZDA,136
222,2024,AUDI,80
226,2024,JEEP,59
229,2024,MINI,45
232,2024,VOLVO,40
224,2024,CADILLAC,35
227,2024,LEXUS,18
221,2024,ALFA ROMEO,12
231,2024,POLESTAR,10


In [34]:
# 33. Top manufacturer in each model year
top_make_each_year = (
    df.groupby(["Model_Year", "Make"])
      .size()
      .reset_index(name="vehicle_records")
      .sort_values(["Model_Year", "vehicle_records"], ascending=[True, False])
      .groupby("Model_Year")
      .head(1)
)

display(top_make_each_year.tail(25))

,Model_Year,Make,vehicle_records
0,1997,CHEVROLET,1
1,1998,FORD,1
2,1999,FORD,4
3,2000,FORD,8
4,2002,TOYOTA,2
5,2003,TOYOTA,1
6,2008,TESLA,19
7,2010,TESLA,21
11,2011,NISSAN,701
19,2012,NISSAN,575


## 8. GroupBy + Percentage / Share Analysis

In [35]:
# 34. Overall manufacturer share
make_share = (
    df["Make"].value_counts()
      .rename("vehicle_records")
      .to_frame()
)

make_share["share_pct"] = (
    make_share["vehicle_records"] / len(df) * 100
).round(2)

display(make_share.head(25))

,vehicle_records,share_pct
Make,,
TESLA,68983,45.84
NISSAN,13497,8.97
CHEVROLET,12026,7.99
FORD,7614,5.06
BMW,6439,4.28
KIA,6198,4.12
TOYOTA,5223,3.47
VOLKSWAGEN,4074,2.71
VOLVO,3536,2.35


In [36]:
# 35. State share
state_share = (
    df["State"].value_counts()
      .rename("vehicle_records")
      .to_frame()
)

state_share["share_pct"] = (
    state_share["vehicle_records"] / len(df) * 100
).round(2)

display(state_share.head(25))

,vehicle_records,share_pct
State,,
WA,150141,99.77
CA,92,0.06
VA,35,0.02
MD,33,0.02
TX,20,0.01
NC,13,0.01
IL,12,0.01
AZ,11,0.01
CO,11,0.01


In [37]:
# 36. EV type share
ev_share = (
    df["EV_Type"].value_counts()
      .rename("vehicle_records")
      .to_frame()
)

ev_share["share_pct"] = (
    ev_share["vehicle_records"] / len(df) * 100
).round(2)

display(ev_share)

,vehicle_records,share_pct
EV_Type,,
Battery Electric Vehicle (BEV),116807,77.62
Plug-in Hybrid Electric Vehicle (PHEV),33675,22.38


In [38]:
# 37. CAFV eligibility share
cafv_share = (
    df["CAFV_Eligibility"].value_counts(dropna=False)
      .rename("vehicle_records")
      .to_frame()
)

cafv_share["share_pct"] = (
    cafv_share["vehicle_records"] / len(df) * 100
).round(2)

display(cafv_share)

,vehicle_records,share_pct
CAFV_Eligibility,,
Eligibility unknown as battery range has not been researched,69698,46.32
Clean Alternative Fuel Vehicle Eligible,62951,41.83
Not eligible due to low battery range,17833,11.85


In [39]:
# 38. Make share using groupby
make_share_groupby = (
    df.groupby("Make")
      .size()
      .div(len(df))
      .mul(100)
      .sort_values(ascending=False)
      .to_frame("share_pct")
)

display(make_share_groupby.head(25))

,share_pct
Make,
TESLA,45.84
NISSAN,8.97
CHEVROLET,7.99
FORD,5.06
BMW,4.28
KIA,4.12
TOYOTA,3.47
VOLKSWAGEN,2.71
VOLVO,2.35


## 9. Pivot Tables

In [40]:
# 39. Basic pivot: State × EV Type
pivot_state_ev = pd.pivot_table(
    df,
    index="State",
    columns="EV_Type",
    values="VIN",
    aggfunc="count",
    fill_value=0
)

display(pivot_state_ev.head(25))

EV_Type,Battery Electric Vehicle (BEV),Plug-in Hybrid Electric Vehicle (PHEV)
State,,
AK,1,0
AL,3,0
AP,1,0
AR,2,0
AZ,6,5
BC,2,0
CA,61,31
CO,9,2
CT,1,6


In [41]:
# 40. Pivot: Make × EV Type
pivot_make_ev = pd.pivot_table(
    df,
    index="Make",
    columns="EV_Type",
    values="VIN",
    aggfunc="count",
    fill_value=0
)

display(pivot_make_ev.head(30))

EV_Type,Battery Electric Vehicle (BEV),Plug-in Hybrid Electric Vehicle (PHEV)
Make,,
ALFA ROMEO,0,12
AUDI,1662,1344
AZURE DYNAMICS,9,0
BENTLEY,0,2
BMW,1796,4643
CADILLAC,84,92
CHEVROLET,7136,4890
CHRYSLER,0,2642
FIAT,806,0


In [42]:
# 41. Pivot: Model Year × EV Type
pivot_year_ev = pd.pivot_table(
    df,
    index="Model_Year",
    columns="EV_Type",
    values="VIN",
    aggfunc="count",
    fill_value=0
)

display(pivot_year_ev.tail(30))

EV_Type,Battery Electric Vehicle (BEV),Plug-in Hybrid Electric Vehicle (PHEV)
Model_Year,,
1997,1,0
1998,1,0
1999,4,0
2000,8,0
2002,2,0
2003,1,0
2008,19,0
2010,21,3
2011,718,78


In [43]:
# 42. Pivot: State × average range
pivot_state_range = pd.pivot_table(
    df,
    index="State",
    columns="EV_Type",
    values="Electric_Range",
    aggfunc="mean",
    fill_value=0
)

display(pivot_state_range.head(25).round(2))

EV_Type,Battery Electric Vehicle (BEV),Plug-in Hybrid Electric Vehicle (PHEV)
State,,
AK,0.00,0.00
AL,73.33,0.00
AP,75.00,0.00
AR,236.00,0.00
AZ,47.17,24.40
BC,220.00,0.00
CA,90.92,26.45
CO,104.89,27.00
CT,322.00,33.17


In [44]:
# 43. Pivot: Make × average MSRP
pivot_make_msrp = pd.pivot_table(
    df,
    index="Make",
    columns="EV_Type",
    values="Base_MSRP",
    aggfunc="mean",
    fill_value=0
)

display(pivot_make_msrp.head(30).round(2))

EV_Type,Battery Electric Vehicle (BEV),Plug-in Hybrid Electric Vehicle (PHEV)
Make,,
ALFA ROMEO,0.00,0.00
AUDI,0.00,0.00
AZURE DYNAMICS,0.00,0.00
BENTLEY,0.00,0.00
BMW,0.00,"5,761.02"
CADILLAC,0.00,"12,243.75"
CHEVROLET,0.00,0.00
CHRYSLER,0.00,"1,771.16"
FIAT,0.00,0.00


In [45]:
# 44. Pivot with multiple aggregations
pivot_multi = pd.pivot_table(
    df,
    index="EV_Type",
    values=["Electric_Range", "Base_MSRP"],
    aggfunc={
        "Electric_Range": ["count", "mean", "median", "max"],
        "Base_MSRP": ["mean", "median", "max"]
    }
)

display(pivot_multi.round(2))

Base_MSRP                  \
                                             max     mean median   
EV_Type                                                            
Battery Electric Vehicle (BEV)            110950 1,146.23   0.00   
Plug-in Hybrid Electric Vehicle (PHEV)    845000 1,889.88   0.00   

                                       Electric_Range                    
                                                count  max  mean median  
EV_Type                                                                  
Battery Electric Vehicle (BEV)                 116807  337 78.61   0.00  
Plug-in Hybrid Electric Vehicle (PHEV)          33675  153 30.66  26.00

In [46]:
# 45. Pivot with margins / grand totals
pivot_margins = pd.pivot_table(
    df,
    index="State",
    columns="EV_Type",
    values="VIN",
    aggfunc="count",
    fill_value=0,
    margins=True,
    margins_name="Grand Total"
)

display(pivot_margins.head(25))

EV_Type,Battery Electric Vehicle (BEV),Plug-in Hybrid Electric Vehicle (PHEV),Grand Total
State,,,
AK,1,0,1
AL,3,0,3
AP,1,0,1
AR,2,0,2
AZ,6,5,11
BC,2,0,2
CA,61,31,92
CO,9,2,11
CT,1,6,7


In [47]:
# 46. Percentage pivot by row
pivot_row_pct = pd.crosstab(
    df["State"],
    df["EV_Type"],
    normalize="index"
).mul(100).round(2)

display(pivot_row_pct.head(25))

EV_Type,Battery Electric Vehicle (BEV),Plug-in Hybrid Electric Vehicle (PHEV)
State,,
AK,100.00,0.00
AL,100.00,0.00
AP,100.00,0.00
AR,100.00,0.00
AZ,54.55,45.45
BC,100.00,0.00
CA,66.30,33.70
CO,81.82,18.18
CT,14.29,85.71


## 10. Cross-Tabulation

In [48]:
# 47. Basic crosstab: Make × EV Type
ct_make_ev = pd.crosstab(df["Make"], df["EV_Type"])
display(ct_make_ev.head(30))

EV_Type,Battery Electric Vehicle (BEV),Plug-in Hybrid Electric Vehicle (PHEV)
Make,,
ALFA ROMEO,0,12
AUDI,1662,1344
AZURE DYNAMICS,9,0
BENTLEY,0,2
BMW,1796,4643
CADILLAC,84,92
CHEVROLET,7136,4890
CHRYSLER,0,2642
FIAT,806,0


In [49]:
# 48. Crosstab with totals
ct_make_ev_totals = pd.crosstab(
    df["Make"],
    df["EV_Type"],
    margins=True,
    margins_name="Total"
)

display(ct_make_ev_totals.head(30))

EV_Type,Battery Electric Vehicle (BEV),Plug-in Hybrid Electric Vehicle (PHEV),Total
Make,,,
ALFA ROMEO,0,12,12
AUDI,1662,1344,3006
AZURE DYNAMICS,9,0,9
BENTLEY,0,2,2
BMW,1796,4643,6439
CADILLAC,84,92,176
CHEVROLET,7136,4890,12026
CHRYSLER,0,2642,2642
FIAT,806,0,806


In [50]:
# 49. Crosstab: State × EV Type
ct_state_ev = pd.crosstab(df["State"], df["EV_Type"])
display(ct_state_ev.head(25))

EV_Type,Battery Electric Vehicle (BEV),Plug-in Hybrid Electric Vehicle (PHEV)
State,,
AK,1,0
AL,3,0
AP,1,0
AR,2,0
AZ,6,5
BC,2,0
CA,61,31
CO,9,2
CT,1,6


In [51]:
# 50. Crosstab: Make × CAFV
ct_make_cafv = pd.crosstab(df["Make"], df["CAFV_Eligibility"])
display(ct_make_cafv.head(30))

CAFV_Eligibility,Clean Alternative Fuel Vehicle Eligible,Eligibility unknown as battery range has not been researched,Not eligible due to low battery range
Make,,,
ALFA ROMEO,12,0,0
AUDI,650,1012,1344
AZURE DYNAMICS,9,0,0
BENTLEY,0,0,2
BMW,3423,1239,1777
CADILLAC,92,84,0
CHEVROLET,8849,3177,0
CHRYSLER,2642,0,0
FIAT,806,0,0


In [52]:
# 51. Crosstab: State × CAFV
ct_state_cafv = pd.crosstab(df["State"], df["CAFV_Eligibility"])
display(ct_state_cafv.head(25))

CAFV_Eligibility,Clean Alternative Fuel Vehicle Eligible,Eligibility unknown as battery range has not been researched,Not eligible due to low battery range
State,,,
AK,0,1,0
AL,1,2,0
AP,1,0,0
AR,2,0,0
AZ,3,4,4
BC,2,0,0
CA,42,33,17
CO,6,4,1
CT,5,0,2


In [53]:
# 52. Crosstab normalized by row
ct_make_ev_row_pct = pd.crosstab(
    df["Make"],
    df["EV_Type"],
    normalize="index"
).mul(100).round(2)

display(ct_make_ev_row_pct.head(30))

EV_Type,Battery Electric Vehicle (BEV),Plug-in Hybrid Electric Vehicle (PHEV)
Make,,
ALFA ROMEO,0.00,100.00
AUDI,55.29,44.71
AZURE DYNAMICS,100.00,0.00
BENTLEY,0.00,100.00
BMW,27.89,72.11
CADILLAC,47.73,52.27
CHEVROLET,59.34,40.66
CHRYSLER,0.00,100.00
FIAT,100.00,0.00


In [54]:
# 53. Crosstab normalized by column
ct_make_ev_col_pct = pd.crosstab(
    df["Make"],
    df["EV_Type"],
    normalize="columns"
).mul(100).round(2)

display(ct_make_ev_col_pct.head(30))

EV_Type,Battery Electric Vehicle (BEV),Plug-in Hybrid Electric Vehicle (PHEV)
Make,,
ALFA ROMEO,0.00,0.04
AUDI,1.42,3.99
AZURE DYNAMICS,0.01,0.00
BENTLEY,0.00,0.01
BMW,1.54,13.79
CADILLAC,0.07,0.27
CHEVROLET,6.11,14.52
CHRYSLER,0.00,7.85
FIAT,0.69,0.00


In [55]:
# 54. Crosstab normalized overall
ct_make_ev_all_pct = pd.crosstab(
    df["Make"],
    df["EV_Type"],
    normalize="all"
).mul(100).round(2)

display(ct_make_ev_all_pct.head(30))

EV_Type,Battery Electric Vehicle (BEV),Plug-in Hybrid Electric Vehicle (PHEV)
Make,,
ALFA ROMEO,0.00,0.01
AUDI,1.10,0.89
AZURE DYNAMICS,0.01,0.00
BENTLEY,0.00,0.00
BMW,1.19,3.09
CADILLAC,0.06,0.06
CHEVROLET,4.74,3.25
CHRYSLER,0.00,1.76
FIAT,0.54,0.00


## 11. Multi-Dimensional Grouping

In [56]:
# 55. State + EV type
state_ev_summary = (
    df.groupby(["State", "EV_Type"])
      .agg(
          vehicle_records=("VIN", "count"),
          unique_makes=("Make", "nunique"),
          unique_models=("Model", "nunique"),
          avg_range=("Electric_Range", "mean"),
          median_range=("Electric_Range", "median"),
          avg_msrp=("Base_MSRP", "mean")
      )
      .sort_values("vehicle_records", ascending=False)
)

display(state_ev_summary.head(40).round(2))

vehicle_records  unique_makes  \
State EV_Type                                                                 
WA    Battery Electric Vehicle (BEV)                   116585            27   
      Plug-in Hybrid Electric Vehicle (PHEV)            33556            25   
CA    Battery Electric Vehicle (BEV)                       61             8   
      Plug-in Hybrid Electric Vehicle (PHEV)               31            12   
VA    Battery Electric Vehicle (BEV)                       20             4   
MD    Battery Electric Vehicle (BEV)                       18             6   
      Plug-in Hybrid Electric Vehicle (PHEV)               15             9   
VA    Plug-in Hybrid Electric Vehicle (PHEV)               15             7   
TX    Battery Electric Vehicle (BEV)                       14             5   
NC    Battery Electric Vehicle (BEV)                        9             1   
IL    Battery Electric Vehicle (BEV)                        9             3   
CO    Battery Electric Vehicle (BEV)                        9             5   
TX    Plug-in Hybrid Electric Vehicle (PHEV)                6             4   
CT    Plug-in Hybrid Electric Vehicle (PHEV)                6             5   
FL    Battery Electric Vehicle (BEV)                        6             1   
AZ    Battery Electric Vehicle (BEV)                        6             3   
      Plug-in Hybrid Electric Vehicle (PHEV)                5             5   
DC    Battery Electric Vehicle (BEV)                        5             2   
NJ    Battery Electric Vehicle (BEV)                        5             2   
NY    Battery Electric Vehicle (BEV)                        5             2   
OR    Plug-in Hybrid Electric Vehicle (PHEV)                5             4   
HI    Plug-in Hybrid Electric Vehicle (PHEV)                4             2   
      Battery Electric Vehicle (BEV)                        4             2   
NV    Battery Electric Vehicle (BEV)                        4             1   
NJ    Plug-in Hybrid Electric Vehicle (PHEV)                4             4   
NC    Plug-in Hybrid Electric Vehicle (PHEV)                4             3   
GA    Battery Electric Vehicle (BEV)                        4             3   
AL    Battery Electric Vehicle (BEV)                        3             2   
OR    Battery Electric Vehicle (BEV)                        3             2   
SC    Battery Electric Vehicle (BEV)                        3             1   
MO    Battery Electric Vehicle (BEV)                        3             1   
FL    Plug-in Hybrid Electric Vehicle (PHEV)                3             3   
IL    Plug-in Hybrid Electric Vehicle (PHEV)                3             3   
LA    Battery Electric Vehicle (BEV)                        3             2   
SC    Plug-in Hybrid Electric Vehicle (PHEV)                3             3   
CO    Plug-in Hybrid Electric Vehicle (PHEV)                2             2   
OH    Battery Electric Vehicle (BEV)                        2             2   
NY    Plug-in Hybrid Electric Vehicle (PHEV)                2             2   
PA    Battery Electric Vehicle (BEV)                        2             1   
WY    Battery Electric Vehicle (BEV)                        2             2   

                                              unique_models  avg_range  \
State EV_Type                                                            
WA    Battery Electric Vehicle (BEV)                     67      78.57   
      Plug-in Hybrid Electric Vehicle (PHEV)             63      30.67   
CA    Battery Electric Vehicle (BEV)                     13      90.92   
      Plug-in Hybrid Electric Vehicle (PHEV)             16      26.45   
VA    Battery Electric Vehicle (BEV)                      8     105.00   
MD    Battery Electric Vehicle (BEV)                      9      78.00   
      Plug-in Hybrid Electric Vehicle (PHEV)             11      32.60   
VA    Plug-in Hybrid Electric Vehicle (PHEV)             10  

In [57]:
# 56. Make + EV type
make_ev_summary = (
    df.groupby(["Make", "EV_Type"])
      .agg(
          vehicle_records=("VIN", "count"),
          unique_models=("Model", "nunique"),
          avg_range=("Electric_Range", "mean"),
          median_range=("Electric_Range", "median"),
          avg_msrp=("Base_MSRP", "mean")
      )
      .sort_values("vehicle_records", ascending=False)
)

display(make_ev_summary.head(50).round(2))

,,vehicle_records,unique_models,avg_range,median_range,avg_msrp
Make,EV_Type,,,,,
TESLA,Battery Electric Vehicle (BEV),68983,5,90.30,0.00,"1,648.60"
NISSAN,Battery Electric Vehicle (BEV),13497,2,84.72,84.00,0.00
CHEVROLET,Battery Electric Vehicle (BEV),7136,4,129.47,238.00,0.00
TOYOTA,Plug-in Hybrid Electric Vehicle (PHEV),5034,4,26.71,25.00,0.00
CHEVROLET,Plug-in Hybrid Electric Vehicle (PHEV),4890,1,45.36,53.00,0.00
BMW,Plug-in Hybrid Electric Vehicle (PHEV),4643,9,41.02,30.00,"5,761.02"
KIA,Battery Electric Vehicle (BEV),4148,4,63.69,0.00,"4,860.74"
VOLKSWAGEN,Battery Electric Vehicle (BEV),4074,2,28.33,0.00,0.00
FORD,Battery Electric Vehicle (BEV),3908,5,5.04,0.00,0.00


In [58]:
# 57. Model year + EV type
year_ev_summary = (
    df.groupby(["Model_Year", "EV_Type"])
      .agg(
          vehicle_records=("VIN", "count"),
          unique_makes=("Make", "nunique"),
          unique_models=("Model", "nunique"),
          avg_range=("Electric_Range", "mean"),
          avg_msrp=("Base_MSRP", "mean")
      )
      .sort_values("vehicle_records", ascending=False)
)

display(year_ev_summary.head(50).round(2))

,,vehicle_records,unique_makes,unique_models,avg_range,avg_msrp
Model_Year,EV_Type,,,,,
2023,Battery Electric Vehicle (BEV),31360,22,45,0.00,0.00
2022,Battery Electric Vehicle (BEV),23511,17,33,0.00,0.00
2021,Battery Electric Vehicle (BEV),14998,14,19,6.49,0.00
2018,Battery Electric Vehicle (BEV),10046,10,12,209.77,250.08
2020,Battery Electric Vehicle (BEV),9617,9,13,277.65,0.00
2019,Battery Electric Vehicle (BEV),8772,11,15,209.11,0.00
2023,Plug-in Hybrid Electric Vehicle (PHEV),5719,15,29,29.04,0.00
2017,Battery Electric Vehicle (BEV),4466,12,14,180.30,"1,097.63"
2018,Plug-in Hybrid Electric Vehicle (PHEV),4395,15,25,33.37,"7,654.82"


In [59]:
# 58. Make + State
make_state_summary = (
    df.groupby(["Make", "State"])
      .size()
      .reset_index(name="vehicle_records")
      .sort_values("vehicle_records", ascending=False)
)

display(make_state_summary.head(50))

,Make,State,vehicle_records
157,TESLA,WA,68821
114,NISSAN,WA,13481
35,CHEVROLET,WA,12004
61,FORD,WA,7592
18,BMW,WA,6426
89,KIA,WA,6187
171,TOYOTA,WA,5196
179,VOLKSWAGEN,WA,4064
186,VOLVO,WA,3529
80,JEEP,WA,3283


In [60]:
# 59. Make + Model
make_model_summary = (
    df.groupby(["Make", "Model"])
      .agg(
          vehicle_records=("VIN", "count"),
          avg_range=("Electric_Range", "mean"),
          median_range=("Electric_Range", "median"),
          avg_msrp=("Base_MSRP", "mean"),
          model_years=("Model_Year", "nunique")
      )
      .sort_values("vehicle_records", ascending=False)
)

display(make_model_summary.head(50).round(2))

vehicle_records  avg_range  median_range  avg_msrp  \
Make       Model                                                                
TESLA      MODEL Y                   28502      23.26          0.00      0.00   
           MODEL 3                   27709     121.38        215.00      0.00   
NISSAN     LEAF                      13187      86.72         84.00      0.00   
TESLA      MODEL S                    7611     183.46        208.00 14,288.86   
CHEVROLET  BOLT EV                    5733     157.58        238.00      0.00   
TESLA      MODEL X                    5114     155.54        200.00      0.00   
CHEVROLET  VOLT                       4890      45.36         53.00      0.00   
VOLKSWAGEN ID.4                       2999       0.00          0.00      0.00   
KIA        NIRO                       2876      80.40         26.00      0.00   
CHRYSLER   PACIFICA                   2642      32.24         32.00  1,771.16   
JEEP       WRANGLER                   2629      21.66         21.00      0.00   
FORD       MUSTANG MACH-E             2619       0.00          0.00      0.00   
TOYOTA     PRIUS PRIME                2527      25.00         25.00      0.00   
BMW        X5                         2068      26.29         30.00      0.00   
           I3                         1917      88.95         81.00      0.00   
FORD       FUSION                     1829      20.38         19.00      0.00   
KIA        EV6                        1644       0.00          0.00      0.00   
HYUNDAI    IONIQ 5                    1638       0.00          0.00      0.00   
FORD       C-MAX                      1546      19.18         19.00      0.00   
TOYOTA     RAV4 PRIME                 1525      42.00         42.00      0.00   
RIVIAN     R1T                        1357       0.00          0.00      0.00   
VOLVO      XC90                       1233      22.28         18.00  4,372.14   
CHEVROLET  BOLT EUV                   1153       0.00          0.00      0.00   
VOLKSWAGEN E-GOLF                     1075     107.38        125.00      0.00   
AUDI       E-TRON                      979     122.70        204.00      0.00   
VOLVO      XC60                        949      24.13         18.00 12,151.95   
TOYOTA     PRIUS PLUG-IN               942       6.00          6.00      0.00   
VOLVO      XC40                        840       0.00          0.00      0.00   
RIVIAN     R1S                         812       0.00          0.00      0.00   
FIAT       500                         806      85.63         87.00      0.00   
HONDA      CLARITY                     801      47.00         47.00      0.00   
MITSUBISHI OUTLANDER                   790      27.38         22.00      0.00   
FORD       F-150                       768       0.00          0.00      0.00   
POLESTAR   PS2                         764      36.29          0.00      0.00   
BMW        I4                          713       0.00          0.00      0.00   
JEEP       GRAND CHEROKEE              663      25.00         25.00      0.00   
HYUNDAI    KONA ELECTRIC               624       0.00          0.00      0.00   
AUDI       Q5 E                        615      19.76         20.00      0.00   
MINI       HARDTOP                     579      23.56          0.00      0.00   
AUDI       A3                          574      16.00         16.00      0.00   
SUBARU     SOLTERRA                    559       0.00          0.00      0.00   
PORSCHE    TAYCAN                      547      74.74          0.00      0.00   
KIA        SORENTO                     526      32.00         32.00      0.00   
BMW        IX                          491       0.00          0.00      0.00   
           330E                        427      17.71         20.00 20,011.94   
KIA        SOUL                        421      93.00         93.00 30,280.40   
BMW        530E                        408      15.90         16.00 37,963.60   
KIA        SPORTAGE                    401      34.00     

## 12. Ranking & Top-N Analysis

In [61]:
# 60. Manufacturer rank
make_rank = (
    df.groupby("Make")
      .size()
      .sort_values(ascending=False)
      .rank(method="dense", ascending=False)
      .astype(int)
      .to_frame("rank")
)

display(make_rank.head(30))

,rank
Make,
TESLA,1
NISSAN,2
CHEVROLET,3
FORD,4
BMW,5
KIA,6
TOYOTA,7
VOLKSWAGEN,8
VOLVO,9


In [62]:
# 61. Top 10 manufacturers
top_10_makes = (
    df.groupby("Make")
      .size()
      .sort_values(ascending=False)
      .head(10)
      .to_frame("vehicle_records")
)

display(top_10_makes)

,vehicle_records
Make,
TESLA,68983
NISSAN,13497
CHEVROLET,12026
FORD,7614
BMW,6439
KIA,6198
TOYOTA,5223
VOLKSWAGEN,4074
VOLVO,3536


In [63]:
# 62. Top models
top_models = (
    df.groupby(["Make", "Model"])
      .size()
      .sort_values(ascending=False)
      .head(25)
      .to_frame("vehicle_records")
)

display(top_models)

vehicle_records
Make       Model                          
TESLA      MODEL Y                   28502
           MODEL 3                   27709
NISSAN     LEAF                      13187
TESLA      MODEL S                    7611
CHEVROLET  BOLT EV                    5733
TESLA      MODEL X                    5114
CHEVROLET  VOLT                       4890
VOLKSWAGEN ID.4                       2999
KIA        NIRO                       2876
CHRYSLER   PACIFICA                   2642
JEEP       WRANGLER                   2629
FORD       MUSTANG MACH-E             2619
TOYOTA     PRIUS PRIME                2527
BMW        X5                         2068
           I3                         1917
FORD       FUSION                     1829
KIA        EV6                        1644
HYUNDAI    IONIQ 5                    1638
FORD       C-MAX                      1546
TOYOTA     RAV4 PRIME                 1525
RIVIAN     R1T                        1357
VOLVO      XC90                       1233
CHEVROLET  BOLT EUV                   1153
VOLKSWAGEN E-GOLF                     1075
AUDI       E-TRON                      979

In [64]:
# 63. Top states
top_states = (
    df.groupby("State")
      .size()
      .sort_values(ascending=False)
      .head(10)
      .to_frame("vehicle_records")
)

display(top_states)

,vehicle_records
State,
WA,150141
CA,92
VA,35
MD,33
TX,20
NC,13
IL,12
CO,11
AZ,11


In [65]:
# 64. Highest median-range manufacturers
top_range_makes = (
    df.groupby("Make")["Electric_Range"]
      .median()
      .sort_values(ascending=False)
      .head(15)
      .to_frame("median_range")
)

display(top_range_makes.round(2))

,median_range
Make,
JAGUAR,234.00
WHEEGO ELECTRIC CARS,100.00
TH!NK,100.00
FIAT,87.00
NISSAN,84.00
SMART,58.00
AZURE DYNAMICS,56.00
CHEVROLET,53.00
HONDA,47.00


## 13. Feature Engineering — Time / Age Features

In [66]:
# 65. Current analysis year
analysis_year = pd.Timestamp.now().year
print("Analysis year:", analysis_year)

Analysis year: 2026


In [67]:
# 66. Vehicle age
df["Vehicle_Age"] = (analysis_year - df["Model_Year"]).clip(lower=0)

display(df[["Model_Year", "Vehicle_Age"]].head())

,Model_Year,Vehicle_Age
0,2020,6
1,2022,4
2,2023,3
3,2018,8
4,2018,8


In [68]:
# 67. Vehicle age group
df["Vehicle_Age_Group"] = pd.cut(
    df["Vehicle_Age"],
    bins=[-1, 2, 5, 10, np.inf],
    labels=["0-2 Years", "3-5 Years", "6-10 Years", "10+ Years"]
)

display(df["Vehicle_Age_Group"].value_counts(dropna=False).to_frame("vehicle_records"))

,vehicle_records
Vehicle_Age_Group,
3-5 Years,83562
6-10 Years,50676
10+ Years,15602
0-2 Years,642


In [69]:
# 68. Model year generation buckets
df["Model_Year_Group"] = pd.cut(
    df["Model_Year"],
    bins=[-np.inf, 2015, 2018, 2021, 2023, np.inf],
    labels=["<=2015", "2016-2018", "2019-2021", "2022-2023", "2024+"]
)

display(df["Model_Year_Group"].value_counts(dropna=False).sort_index().to_frame("vehicle_records"))

,vehicle_records
Model_Year_Group,
<=2015,15602
2016-2018,28665
2019-2021,40695
2022-2023,64878
2024+,642


## 14. Feature Engineering — Range

In [70]:
# 69. Range availability flag
df["Has_Electric_Range"] = np.where(
    df["Electric_Range"].fillna(0) > 0, 1, 0
)

display(df["Has_Electric_Range"].value_counts().to_frame("vehicle_records"))

,vehicle_records
Has_Electric_Range,
1,80784
0,69698


In [71]:
# 70. Range group
df["Range_Group"] = pd.cut(
    df["Electric_Range"],
    bins=[-np.inf, 0, 50, 100, 200, 300, np.inf],
    labels=[
        "0",
        "1-50",
        "51-100",
        "101-200",
        "201-300",
        "300+"
    ]
)

display(df["Range_Group"].value_counts(dropna=False).sort_index().to_frame("vehicle_records"))

,vehicle_records
Range_Group,
0,69698
1-50,29844
51-100,13658
101-200,6613
201-300,28010
300+,2659


In [72]:
# 71. Range percentile
df["Range_Percentile"] = df["Electric_Range"].rank(pct=True) * 100

display(df[["Electric_Range", "Range_Percentile"]].head())

,Electric_Range,Range_Percentile
0,258,94.01
1,25,55.98
2,25,55.98
3,215,85.10
4,97,75.06


In [73]:
# 72. Range z-score
range_mean = df["Electric_Range"].mean()
range_std = df["Electric_Range"].std()

df["Range_ZScore"] = np.where(
    range_std != 0,
    (df["Electric_Range"] - range_mean) / range_std,
    0
)

display(df[["Electric_Range", "Range_ZScore"]].head())

,Electric_Range,Range_ZScore
0,258,1.98
1,25,-0.45
2,25,-0.45
3,215,1.53
4,97,0.30


## 15. Feature Engineering — MSRP

In [74]:
# 73. MSRP availability flag
df["Has_MSRP"] = np.where(
    df["Base_MSRP"].fillna(0) > 0, 1, 0
)

display(df["Has_MSRP"].value_counts().to_frame("vehicle_records"))

,vehicle_records
Has_MSRP,
0,147027
1,3455


In [75]:
# 74. MSRP group
df["MSRP_Group"] = pd.cut(
    df["Base_MSRP"],
    bins=[-np.inf, 0, 30000, 50000, 75000, 100000, np.inf],
    labels=[
        "0",
        "<=30K",
        "30K-50K",
        "50K-75K",
        "75K-100K",
        "100K+"
    ]
)

display(df["MSRP_Group"].value_counts(dropna=False).sort_index().to_frame("vehicle_records"))

,vehicle_records
MSRP_Group,
0,147027
<=30K,0
30K-50K,1154
50K-75K,2161
75K-100K,83
100K+,57


In [76]:
# 75. MSRP percentile
df["MSRP_Percentile"] = df["Base_MSRP"].rank(pct=True) * 100

display(df[["Base_MSRP", "MSRP_Percentile"]].head())

,Base_MSRP,MSRP_Percentile
0,0,48.85
1,0,48.85
2,0,48.85
3,0,48.85
4,0,48.85


In [77]:
# 76. MSRP z-score
msrp_mean = df["Base_MSRP"].mean()
msrp_std = df["Base_MSRP"].std()

df["MSRP_ZScore"] = np.where(
    msrp_std != 0,
    (df["Base_MSRP"] - msrp_mean) / msrp_std,
    0
)

display(df[["Base_MSRP", "MSRP_ZScore"]].head())

,Base_MSRP,MSRP_ZScore
0,0,-0.14
1,0,-0.14
2,0,-0.14
3,0,-0.14
4,0,-0.14


## 16. Feature Engineering — Range / MSRP Relationship

In [78]:
# 77. Range per MSRP unit
df["Range_per_MSRP"] = np.where(
    df["Base_MSRP"] > 0,
    df["Electric_Range"] / df["Base_MSRP"],
    np.nan
)

display(
    df[["Electric_Range", "Base_MSRP", "Range_per_MSRP"]]
    .head()
)

,Electric_Range,Base_MSRP,Range_per_MSRP
0,258,0,NaN
1,25,0,NaN
2,25,0,NaN
3,215,0,NaN
4,97,0,NaN


In [79]:
# 78. MSRP per range unit
df["MSRP_per_Range"] = np.where(
    df["Electric_Range"] > 0,
    df["Base_MSRP"] / df["Electric_Range"],
    np.nan
)

display(
    df[["Electric_Range", "Base_MSRP", "MSRP_per_Range"]]
    .head()
)

,Electric_Range,Base_MSRP,MSRP_per_Range
0,258,0,0.00
1,25,0,0.00
2,25,0,0.00
3,215,0,0.00
4,97,0,0.00


In [80]:
# 79. Range-to-MSRP group
df["Range_Value_Group"] = pd.qcut(
    df["Range_per_MSRP"].replace([np.inf, -np.inf], np.nan),
    q=4,
    labels=["Low", "Medium-Low", "Medium-High", "High"],
    duplicates="drop"
)

display(df["Range_Value_Group"].value_counts(dropna=False).sort_index().to_frame("vehicle_records"))

,vehicle_records
Range_Value_Group,
Low,956
Medium-Low,847
Medium-High,1441
High,211
NaN,147027


## 17. Feature Engineering — Manufacturer / Geographic Shares

In [81]:
# 80. Manufacturer share
make_counts = df["Make"].value_counts()
df["Make_Share_Pct"] = df["Make"].map(make_counts.div(len(df)).mul(100))

display(df[["Make", "Make_Share_Pct"]].head())

,Make,Make_Share_Pct
0,HYUNDAI,2.11
1,JEEP,2.19
2,JEEP,2.19
3,TESLA,45.84
4,BMW,4.28


In [82]:
# 81. State share
state_counts = df["State"].value_counts()
df["State_Share_Pct"] = df["State"].map(state_counts.div(len(df)).mul(100))

display(df[["State", "State_Share_Pct"]].head())

,State,State_Share_Pct
0,WA,99.77
1,WA,99.77
2,WA,99.77
3,WA,99.77
4,WA,99.77


In [83]:
# 82. EV type share
ev_type_counts = df["EV_Type"].value_counts()
df["EV_Type_Share_Pct"] = df["EV_Type"].map(
    ev_type_counts.div(len(df)).mul(100)
)

display(df[["EV_Type", "EV_Type_Share_Pct"]].head())

,EV_Type,EV_Type_Share_Pct
0,Battery Electric Vehicle (BEV),77.62
1,Plug-in Hybrid Electric Vehicle (PHEV),22.38
2,Plug-in Hybrid Electric Vehicle (PHEV),22.38
3,Battery Electric Vehicle (BEV),77.62
4,Plug-in Hybrid Electric Vehicle (PHEV),22.38


In [84]:
# 83. CAFV share
cafv_counts = df["CAFV_Eligibility"].value_counts(dropna=False)
df["CAFV_Share_Pct"] = df["CAFV_Eligibility"].map(
    cafv_counts.div(len(df)).mul(100)
)

display(df[["CAFV_Eligibility", "CAFV_Share_Pct"]].head())

,CAFV_Eligibility,CAFV_Share_Pct
0,Clean Alternative Fuel Vehicle Eligible,41.83
1,Not eligible due to low battery range,11.85
2,Not eligible due to low battery range,11.85
3,Clean Alternative Fuel Vehicle Eligible,41.83
4,Clean Alternative Fuel Vehicle Eligible,41.83


## 18. Feature Engineering — Ranks

In [85]:
# 84. Manufacturer rank
make_rank_map = df["Make"].value_counts().rank(method="dense", ascending=False)
df["Make_Rank"] = df["Make"].map(make_rank_map)

display(df[["Make", "Make_Rank"]].head(15))

,Make,Make_Rank
0,HYUNDAI,11.00
1,JEEP,10.00
2,JEEP,10.00
3,TESLA,1.00
4,BMW,5.00
5,TESLA,1.00
6,CHRYSLER,13.00
7,TESLA,1.00
8,TESLA,1.00
9,TESLA,1.00


In [86]:
# 85. State rank
state_rank_map = df["State"].value_counts().rank(method="dense", ascending=False)
df["State_Rank"] = df["State"].map(state_rank_map)

display(df[["State", "State_Rank"]].head(15))

,State,State_Rank
0,WA,1.00
1,WA,1.00
2,WA,1.00
3,WA,1.00
4,WA,1.00
5,WA,1.00
6,WA,1.00
7,WA,1.00
8,WA,1.00
9,WA,1.00


In [87]:
# 86. Model rank
model_rank_map = df["Model"].value_counts().rank(method="dense", ascending=False)
df["Model_Rank"] = df["Model"].map(model_rank_map)

display(df[["Model", "Model_Rank"]].head(15))

,Model,Model_Rank
0,KONA,63.00
1,GRAND CHEROKEE,36.00
2,GRAND CHEROKEE,36.00
3,MODEL 3,2.00
4,I3,15.00
5,MODEL 3,2.00
6,PACIFICA,10.00
7,MODEL Y,1.00
8,MODEL 3,2.00
9,MODEL Y,1.00


## 19. Feature Engineering — Binary / Rule-Based Indicators

In [88]:
# 87. Long-range vehicle flag
df["Long_Range_Flag"] = np.where(
    df["Electric_Range"] >= 200, 1, 0
)

display(df["Long_Range_Flag"].value_counts().to_frame("vehicle_records"))

,vehicle_records
Long_Range_Flag,
0,118487
1,31995


In [89]:
# 88. Premium MSRP flag
df["Premium_MSRP_Flag"] = np.where(
    df["Base_MSRP"] >= 75000, 1, 0
)

display(df["Premium_MSRP_Flag"].value_counts().to_frame("vehicle_records"))

,vehicle_records
Premium_MSRP_Flag,
0,150342
1,140


In [90]:
# 89. Recent model flag
df["Recent_Model_Flag"] = np.where(
    df["Model_Year"] >= analysis_year - 3, 1, 0
)

display(df["Recent_Model_Flag"].value_counts().to_frame("vehicle_records"))

,vehicle_records
Recent_Model_Flag,
0,112761
1,37721


In [91]:
# 90. Older model flag
df["Older_Model_Flag"] = np.where(
    df["Model_Year"] <= analysis_year - 10, 1, 0
)

display(df["Older_Model_Flag"].value_counts().to_frame("vehicle_records"))

,vehicle_records
Older_Model_Flag,
0,129230
1,21252


In [92]:
# 91. High-range + premium indicator
df["High_Range_Premium_Flag"] = np.where(
    (df["Electric_Range"] >= 200) &
    (df["Base_MSRP"] >= 75000),
    1,
    0
)

display(df["High_Range_Premium_Flag"].value_counts().to_frame("vehicle_records"))

,vehicle_records
High_Range_Premium_Flag,
0,150435
1,47


## 20. Advanced Analytical Summary Tables

In [93]:
# 92. Manufacturer business-style scorecard
manufacturer_scorecard = (
    df.groupby("Make")
      .agg(
          records=("VIN", "count"),
          models=("Model", "nunique"),
          states=("State", "nunique"),
          avg_range=("Electric_Range", "mean"),
          median_range=("Electric_Range", "median"),
          avg_msrp=("Base_MSRP", "mean"),
          median_msrp=("Base_MSRP", "median"),
          long_range_pct=("Long_Range_Flag", "mean"),
          premium_pct=("Premium_MSRP_Flag", "mean")
      )
)

manufacturer_scorecard["record_share_pct"] = (
    manufacturer_scorecard["records"] / len(df) * 100
)

manufacturer_scorecard["long_range_pct"] *= 100
manufacturer_scorecard["premium_pct"] *= 100

display(
    manufacturer_scorecard
    .sort_values("records", ascending=False)
    .head(30)
    .round(2)
)

,records,models,states,avg_range,median_range,avg_msrp,median_msrp,long_range_pct,premium_pct,record_share_pct
Make,,,,,,,,,,
TESLA,68983,5,35,90.30,0.00,"1,648.60",0.00,37.54,0.07,45.84
NISSAN,13497,2,13,84.72,84.00,0.00,0.00,1.80,0.00,8.97
CHEVROLET,12026,5,14,95.27,53.00,0.00,0.00,30.84,0.00,7.99
FORD,7614,8,12,13.02,19.00,0.00,0.00,0.00,0.00,5.06
BMW,6439,11,11,38.16,30.00,"4,154.12",0.00,0.00,0.47,4.28
KIA,6198,7,9,52.62,26.00,"3,253.04",0.00,13.68,0.00,4.12
TOYOTA,5223,6,12,26.92,25.00,0.00,0.00,0.00,0.00,3.47
VOLKSWAGEN,4074,2,8,28.33,0.00,0.00,0.00,0.00,0.00,2.71
VOLVO,3536,7,7,15.63,18.00,"4,785.93",0.00,0.00,0.00,2.35


In [94]:
# 93. State analytical scorecard
state_scorecard = (
    df.groupby("State")
      .agg(
          records=("VIN", "count"),
          makes=("Make", "nunique"),
          models=("Model", "nunique"),
          avg_range=("Electric_Range", "mean"),
          median_range=("Electric_Range", "median"),
          avg_msrp=("Base_MSRP", "mean"),
          long_range_pct=("Long_Range_Flag", "mean"),
          premium_pct=("Premium_MSRP_Flag", "mean")
      )
)

state_scorecard["record_share_pct"] = (
    state_scorecard["records"] / len(df) * 100
)

state_scorecard["long_range_pct"] *= 100
state_scorecard["premium_pct"] *= 100

display(
    state_scorecard
    .sort_values("records", ascending=False)
    .head(30)
    .round(2)
)

,records,makes,models,avg_range,median_range,avg_msrp,long_range_pct,premium_pct,record_share_pct
State,,,,,,,,,
WA,150141,37,127,67.86,18.00,"1,311.28",21.25,0.09,99.77
CA,92,16,28,69.20,22.00,"1,998.91",23.91,0.00,0.06
VA,35,10,18,73.11,32.00,0.00,22.86,0.00,0.02
MD,33,14,20,57.36,30.00,"3,090.91",18.18,3.03,0.02
TX,20,8,12,80.35,23.00,"1,749.75",30.00,0.00,0.01
NC,13,4,6,40.69,0.00,0.00,15.38,0.00,0.01
IL,12,6,7,94.25,30.50,"3,332.92",33.33,0.00,0.01
CO,11,6,10,90.73,32.00,0.00,27.27,0.00,0.01
AZ,11,8,10,36.82,22.00,0.00,9.09,0.00,0.01


In [95]:
# 94. EV type scorecard
ev_scorecard = (
    df.groupby("EV_Type")
      .agg(
          records=("VIN", "count"),
          makes=("Make", "nunique"),
          models=("Model", "nunique"),
          states=("State", "nunique"),
          avg_range=("Electric_Range", "mean"),
          median_range=("Electric_Range", "median"),
          avg_msrp=("Base_MSRP", "mean"),
          median_msrp=("Base_MSRP", "median")
      )
)

ev_scorecard["share_pct"] = ev_scorecard["records"] / len(df) * 100

display(ev_scorecard.sort_values("records", ascending=False).round(2))

,records,makes,models,states,avg_range,median_range,avg_msrp,median_msrp,share_pct
EV_Type,,,,,,,,,
Battery Electric Vehicle (BEV),116807,27,67,40,78.61,0.00,"1,146.23",0.00,77.62
Plug-in Hybrid Electric Vehicle (PHEV),33675,25,63,26,30.66,26.00,"1,889.88",0.00,22.38


## 21. Feature Engineering Quality Checks

In [96]:
# 95. Check engineered columns
engineered_cols = [
    "Vehicle_Age", "Vehicle_Age_Group", "Model_Year_Group",
    "Has_Electric_Range", "Range_Group", "Range_Percentile",
    "Range_ZScore", "Has_MSRP", "MSRP_Group", "MSRP_Percentile",
    "MSRP_ZScore", "Range_per_MSRP", "MSRP_per_Range",
    "Range_Value_Group", "Make_Share_Pct", "State_Share_Pct",
    "EV_Type_Share_Pct", "CAFV_Share_Pct", "Make_Rank",
    "State_Rank", "Model_Rank", "Long_Range_Flag",
    "Premium_MSRP_Flag", "Recent_Model_Flag", "Older_Model_Flag",
    "High_Range_Premium_Flag"
]

available_engineered = [c for c in engineered_cols if c in df.columns]

display(pd.DataFrame({
    "column": available_engineered,
    "dtype": [df[c].dtype for c in available_engineered],
    "missing": [df[c].isna().sum() for c in available_engineered],
    "missing_pct": [
        round(df[c].isna().mean()*100, 2)
        for c in available_engineered
    ],
    "unique": [df[c].nunique(dropna=False) for c in available_engineered]
}))

,column,dtype,missing,missing_pct,unique
0,Vehicle_Age,int64,0,0.00,22
1,Vehicle_Age_Group,category,0,0.00,4
2,Model_Year_Group,category,0,0.00,5
3,Has_Electric_Range,int64,0,0.00,2
4,Range_Group,category,0,0.00,6
5,Range_Percentile,float64,0,0.00,102
6,Range_ZScore,float64,0,0.00,102
7,Has_MSRP,int64,0,0.00,2
8,MSRP_Group,category,0,0.00,5
9,MSRP_Percentile,float64,0,0.00,31


In [97]:
# 96. Validate binary flags
binary_cols = [
    "Has_Electric_Range",
    "Has_MSRP",
    "Long_Range_Flag",
    "Premium_MSRP_Flag",
    "Recent_Model_Flag",
    "Older_Model_Flag",
    "High_Range_Premium_Flag"
]

for col in binary_cols:
    if col in df.columns:
        print(f"{col}: {sorted(df[col].dropna().unique().tolist())}")

Has_Electric_Range: [0, 1]
Has_MSRP: [0, 1]
Long_Range_Flag: [0, 1]
Premium_MSRP_Flag: [0, 1]
Recent_Model_Flag: [0, 1]
Older_Model_Flag: [0, 1]
High_Range_Premium_Flag: [0, 1]


In [98]:
# 97. Check infinite values in engineered numeric columns
engineered_numeric = df[[
    c for c in [
        "Range_per_MSRP",
        "MSRP_per_Range",
        "Range_ZScore",
        "MSRP_ZScore",
        "Range_Percentile",
        "MSRP_Percentile"
    ] if c in df.columns
]]

display(
    pd.DataFrame({
        "infinite_count": np.isinf(engineered_numeric).sum(),
        "missing_count": engineered_numeric.isna().sum()
    })
)

,infinite_count,missing_count
Range_per_MSRP,0,147027
MSRP_per_Range,0,69698
Range_ZScore,0,0
MSRP_ZScore,0,0
Range_Percentile,0,0
MSRP_Percentile,0,0


In [99]:
# ------------------------------------------------------------
# 1. Clean column names
# ------------------------------------------------------------
df.columns = df.columns.str.strip()

# ------------------------------------------------------------
# 2. Convert numeric columns
# ------------------------------------------------------------
numeric_cols = [
    "Model_Year",
    "Postal_Code",
    "Electric_Range",
    "Base_MSRP",
    "Vehicle_ID",
    "Census_Tract",
    "Legislative_District"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [100]:
# ============================================================
# A. VEHICLE AGE / TIME FEATURES
# ============================================================

CURRENT_YEAR = pd.Timestamp.today().year

# Vehicle age
df["Vehicle_Age"] = CURRENT_YEAR - df["Model_Year"]

# Prevent negative ages caused by future model years
df["Vehicle_Age"] = df["Vehicle_Age"].clip(lower=0)

# Model-year decade
df["Model_Year_Decade"] = (
    (df["Model_Year"] // 10) * 10
).astype("Int64")

# EV generation / era
df["EV_Era"] = pd.cut(
    df["Model_Year"],
    bins=[-np.inf, 2015, 2019, 2022, 2025, np.inf],
    labels=[
        "Early EV",
        "Growing EV",
        "Mainstream EV",
        "Current EV",
        "Future/Latest"
    ]
)


# ============================================================
# B. RANGE FEATURES
# ============================================================

# Range category
df["Range_Category"] = pd.cut(
    df["Electric_Range"],
    bins=[-np.inf, 100, 200, 300, 400, np.inf],
    labels=[
        "Very Low",
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

# Range score
df["Range_Score"] = pd.cut(
    df["Electric_Range"],
    bins=[-np.inf, 100, 200, 300, 400, np.inf],
    labels=[1, 2, 3, 4, 5]
).astype("Int64")


# ============================================================
# C. PRICE FEATURES
# ============================================================

# MSRP category
df["Price_Category"] = pd.cut(
    df["Base_MSRP"],
    bins=[-np.inf, 30000, 50000, 80000, 120000, np.inf],
    labels=[
        "Budget",
        "Mid-Range",
        "Premium",
        "Luxury",
        "Ultra Luxury"
    ]
)

# Log price — useful for statistical analysis
df["Log_MSRP"] = np.log1p(df["Base_MSRP"])


# ============================================================
# D. RANGE / PRICE BUSINESS METRIC
# ============================================================

# Range per $10,000 MSRP
df["Range_per_10K_MSRP"] = np.where(
    df["Base_MSRP"] > 0,
    df["Electric_Range"] / (df["Base_MSRP"] / 10000),
    np.nan
)


# ============================================================
# E. EV TYPE FEATURES
# ============================================================

# Binary EV type indicator
df["Is_BEV"] = (
    df["EV_Type"]
    .astype(str)
    .str.upper()
    .str.contains("BATTERY|BEV")
    .astype(int)
)

df["Is_PHEV"] = (
    df["EV_Type"]
    .astype(str)
    .str.upper()
    .str.contains("PLUGIN|PHEV|PLUG-IN")
    .astype(int)
)


# ============================================================
# F. MAKE / MANUFACTURER FEATURES
# ============================================================

# Manufacturer frequency
make_frequency = df["Make"].value_counts()

df["Manufacturer_Frequency"] = (
    df["Make"].map(make_frequency)
)

# Manufacturer market size group
df["Manufacturer_Size"] = pd.cut(
    df["Manufacturer_Frequency"],
    bins=[-np.inf, 10, 100, 1000, np.inf],
    labels=[
        "Small",
        "Medium",
        "Large",
        "Very Large"
    ]
)


# ============================================================
# G. MODEL FEATURES
# ============================================================

# Number of vehicles/records for each model
model_frequency = df["Model"].value_counts()

df["Model_Frequency"] = (
    df["Model"].map(model_frequency)
)

# Model popularity category
df["Model_Popularity"] = pd.cut(
    df["Model_Frequency"],
    bins=[-np.inf, 10, 100, 1000, np.inf],
    labels=[
        "Rare",
        "Moderate",
        "Popular",
        "Highly Popular"
    ]
)


# ============================================================
# H. LOCATION FEATURES
# ============================================================

# Geographic hierarchy
df["Location_Key"] = (
    df["State"].astype(str).str.strip()
    + "_"
    + df["County"].astype(str).str.strip()
)

# State-level vehicle count
state_frequency = df["State"].value_counts()

df["State_EV_Count"] = (
    df["State"].map(state_frequency)
)

# City-level vehicle count
city_frequency = df["City"].value_counts()

df["City_EV_Count"] = (
    df["City"].map(city_frequency)
)


# ============================================================
# I. UTILITY FEATURES
# ============================================================

utility_frequency = df["Electric_Utility"].value_counts()

df["Utility_EV_Count"] = (
    df["Electric_Utility"].map(utility_frequency)
)


# ============================================================
# J. CAFV FEATURES
# ============================================================

df["CAFV_Eligible_Flag"] = (
    df["CAFV_Eligibility"]
    .astype(str)
    .str.upper()
    .str.contains("ELIGIBLE")
    .astype(int)
)

df["CAFV_Status"] = (
    df["CAFV_Eligibility"]
    .astype(str)
    .str.strip()
)


# ============================================================
# K. LOCATION DATA QUALITY FEATURES
# ============================================================

df["Has_Location"] = (
    df["Vehicle_Location"]
    .notna()
    .astype(int)
)

df["Has_Census_Tract"] = (
    df["Census_Tract"]
    .notna()
    .astype(int)
)

df["Has_Legislative_District"] = (
    df["Legislative_District"]
    .notna()
    .astype(int)
)


# ============================================================
# L. DATA QUALITY / IDENTIFIER FEATURES
# ============================================================

# VIN available?
df["Has_VIN"] = (
    df["VIN"]
    .notna()
    .astype(int)
)

# Missing-value count per row
df["Missing_Features_Count"] = (
    df.isna().sum(axis=1)
)

# Data completeness percentage
df["Data_Completeness_Pct"] = (
    1 - df.isna().sum(axis=1) / df.shape[1]
) * 100

In [101]:
# ============================================================
# FINAL CHECK
# ============================================================

print("Original columns:", len([
    "VIN", "County", "City", "State", "Postal_Code",
    "Model_Year", "Make", "Model", "EV_Type",
    "CAFV_Eligibility", "Electric_Range", "Base_MSRP",
    "Vehicle_ID", "Vehicle_Location", "Electric_Utility",
    "Census_Tract", "Legislative_District"
]))

print("Final shape:", df.shape)

print("\nNew engineered columns:")
new_features = [
    "Vehicle_Age",
    "Model_Year_Decade",
    "EV_Era",
    "Range_Category",
    "Range_Score",
    "Price_Category",
    "Log_MSRP",
    "Range_per_10K_MSRP",
    "Is_BEV",
    "Is_PHEV",
    "Manufacturer_Frequency",
    "Manufacturer_Size",
    "Model_Frequency",
    "Model_Popularity",
    "Location_Key",
    "State_EV_Count",
    "City_EV_Count",
    "Utility_EV_Count",
    "CAFV_Eligible_Flag",
    "CAFV_Status",
    "Has_Location",
    "Has_Census_Tract",
    "Has_Legislative_District",
    "Has_VIN",
    "Missing_Features_Count",
    "Data_Completeness_Pct"
]

for col in new_features:
    print("✔", col)

display(df.head())

Original columns: 17
Final shape: (150482, 68)

New engineered columns:
✔ Vehicle_Age
✔ Model_Year_Decade
✔ EV_Era
✔ Range_Category
✔ Range_Score
✔ Price_Category
✔ Log_MSRP
✔ Range_per_10K_MSRP
✔ Is_BEV
✔ Is_PHEV
✔ Manufacturer_Frequency
✔ Manufacturer_Size
✔ Model_Frequency
✔ Model_Popularity
✔ Location_Key
✔ State_EV_Count
✔ City_EV_Count
✔ Utility_EV_Count
✔ CAFV_Eligible_Flag
✔ CAFV_Status
✔ Has_Location
✔ Has_Census_Tract
✔ Has_Legislative_District
✔ Has_VIN
✔ Missing_Features_Count
✔ Data_Completeness_Pct


,VIN,County,City,State,Postal_Code,Model_Year,Make,Model,EV_Type,CAFV_Eligibility,Electric_Range,Base_MSRP,Vehicle_ID,Vehicle_Location,Electric_Utility,Census_Tract,Legislative_District,Vehicle_Age,Vehicle_Age_Group,Model_Year_Group,Has_Electric_Range,Range_Group,Range_Percentile,Range_ZScore,Has_MSRP,MSRP_Group,MSRP_Percentile,MSRP_ZScore,Range_per_MSRP,MSRP_per_Range,Range_Value_Group,Make_Share_Pct,State_Share_Pct,EV_Type_Share_Pct,CAFV_Share_Pct,Make_Rank,State_Rank,Model_Rank,Long_Range_Flag,Premium_MSRP_Flag,Recent_Model_Flag,Older_Model_Flag,High_Range_Premium_Flag,Model_Year_Decade,EV_Era,Range_Category,Range_Score,Price_Category,Log_MSRP,Range_per_10K_MSRP,Is_BEV,Is_PHEV,Manufacturer_Frequency,Manufacturer_Size,Model_Frequency,Model_Popularity,Location_Key,State_EV_Count,City_EV_Count,Utility_EV_Count,CAFV_Eligible_Flag,CAFV_Status,Has_Location,Has_Census_Tract,Has_Legislative_District,Has_VIN,Missing_Features_Count,Data_Completeness_Pct
0,KM8K33AGXL,King,Seattle,WA,"98,103.00",2020,HYUNDAI,KONA,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,258,0,249675142,POINT (-122.34301 47.659185),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),"53,033,004,800.00",43,6,6-10 Years,2019-2021,1,201-300,94.01,1.98,0,0,48.85,-0.14,NaN,0.00,NaN,2.11,99.77,77.62,41.83,11.00,1.00,63.00,1,0,0,0,0,2020,Mainstream EV,Medium,3,Budget,0.00,NaN,1,0,3171,Very Large,208,Popular,WA_King,150141,25678,27268,1,Clean Alternative Fuel Vehicle Eligible,1,1,1,1,3,95.52
1,1C4RJYB61N,King,Bothell,WA,"98,011.00",2022,JEEP,GRAND CHEROKEE,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,25,0,233928502,POINT (-122.20578 47.762405),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),"53,033,021,804.00",1,4,3-5 Years,2022-2023,1,1-50,55.98,-0.45,0,0,48.85,-0.14,NaN,0.00,NaN,2.19,99.77,22.38,11.85,10.00,1.00,36.00,0,0,0,0,0,2020,Mainstream EV,Very Low,1,Budget,0.00,NaN,0,1,3292,Very Large,663,Popular,WA_King,150141,4861,55637,1,Not eligible due to low battery range,1,1,1,1,3,95.52
2,1C4RJYD61P,Yakima,Yakima,WA,"98,908.00",2023,JEEP,GRAND CHEROKEE,Plug-in Hybrid Electric Vehicle (PHEV),Not eligible due to low battery range,25,0,229675939,POINT (-120.6027202 46.5965625),PACIFICORP,"53,077,002,900.00",14,3,3-5 Years,2022-2023,1,1-50,55.98,-0.45,0,0,48.85,-0.14,NaN,0.00,NaN,2.19,99.77,22.38,11.85,10.00,1.00,36.00,0,0,1,0,0,2020,Current EV,Very Low,1,Budget,0.00,NaN,0,1,3292,Very Large,663,Popular,WA_Yakima,150141,552,1102,1,Not eligible due to low battery range,1,1,1,1,3,95.52
3,5YJ3E1EA7J,King,Kirkland,WA,"98,034.00",2018,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,215,0,104714466,POINT (-122.209285 47.71124),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),"53,033,021,903.00",45,8,6-10 Years,2016-2018,1,201-300,85.10,1.53,0,0,48.85,-0.14,NaN,0.00,NaN,45.84,99.77,77.62,41.83,1.00,1.00,2.00,1,0,0,0,0,2010,Growing EV,Medium,3,Budget,0.00,NaN,1,0,68983,Very Large,27709,Highly Popular,WA_King,150141,4622,55637,1,Clean Alternative Fuel Vehicle Eligible,1,1,1,1,3,95.52
4,WBY7Z8C5XJ,Thurston,Olympia,WA,"98,501.00",2018,BMW,I3,Plug-in Hybrid Electric Vehicle (PHEV),Clean Alternative Fuel Vehicle Eligible,97,0,185498386,POINT (-122.89692 47.043535),PUGET SOUND ENERGY INC,"53,067,010,700.00",22,8,6-10 Years,2016-2018,1,51-100,75.06,0.30,0,0,48.85,-0.14,NaN,0.00,NaN,4.28,99.77,22.38,41.83,5.00,1.00,15.00,0,0,0,0,0,2010,Growing EV,Very Low,1,Budget,0.00,NaN,0,1,6439,Very Large,1917,Highly Popular,WA_Thurston,150141,3634,29865,1,Clean Alternative Fuel Vehicle Eligible,1,1,1,1,3,95.52


In [102]:
df.columns

Index(['VIN', 'County', 'City', 'State', 'Postal_Code', 'Model_Year', 'Make',
       'Model', 'EV_Type', 'CAFV_Eligibility', 'Electric_Range', 'Base_MSRP',
       'Vehicle_ID', 'Vehicle_Location', 'Electric_Utility', 'Census_Tract',
       'Legislative_District', 'Vehicle_Age', 'Vehicle_Age_Group',
       'Model_Year_Group', 'Has_Electric_Range', 'Range_Group',
       'Range_Percentile', 'Range_ZScore', 'Has_MSRP', 'MSRP_Group',
       'MSRP_Percentile', 'MSRP_ZScore', 'Range_per_MSRP', 'MSRP_per_Range',
       'Range_Value_Group', 'Make_Share_Pct', 'State_Share_Pct',
       'EV_Type_Share_Pct', 'CAFV_Share_Pct', 'Make_Rank', 'State_Rank',
       'Model_Rank', 'Long_Range_Flag', 'Premium_MSRP_Flag',
       'Recent_Model_Flag', 'Older_Model_Flag', 'High_Range_Premium_Flag',
       'Model_Year_Decade', 'EV_Era', 'Range_Category', 'Range_Score',
       'Price_Category', 'Log_MSRP', 'Range_per_10K_MSRP', 'Is_BEV', 'Is_PHEV',
       'Manufacturer_Frequency', 'Manufacturer_Size', 'Model_Freq

In [110]:
len(df.columns)

68

## 22. Save Analytical Dataset & Summary Tables

In [111]:
# 98. Create output folders
from pathlib import Path

output_dir = Path("../datas/processed")
table_dir = Path("../outputs/tables")

output_dir.mkdir(parents=True, exist_ok=True)
table_dir.mkdir(parents=True, exist_ok=True)

print("✅ Output folders ready.")

✅ Output folders ready.


In [112]:
# 99. Save feature-engineered dataset
processed_path = output_dir / "EV_Analytical_Feature_Engineered.csv"
df.to_csv(processed_path, index=False)

print(f"✅ Saved: {processed_path}")

✅ Saved: ..\datas\processed\EV_Analytical_Feature_Engineered.csv


In [113]:
# 100. Save major summary tables
tables = {
    "manufacturer_scorecard.csv": manufacturer_scorecard.reset_index(),
    "state_scorecard.csv": state_scorecard.reset_index(),
    "ev_type_scorecard.csv": ev_scorecard.reset_index(),
    "year_profile.csv": year_profile.reset_index(),
    "make_profile.csv": make_profile.reset_index(),
    "state_profile.csv": state_profile.reset_index(),
    "make_ev_summary.csv": make_ev_summary.reset_index(),
    "state_ev_summary.csv": state_ev_summary.reset_index(),
    "pivot_state_ev.csv": pivot_state_ev.reset_index(),
    "pivot_make_ev.csv": pivot_make_ev.reset_index(),
    "pivot_year_ev.csv": pivot_year_ev.reset_index(),
    "ct_make_ev.csv": ct_make_ev.reset_index(),
    "ct_state_ev.csv": ct_state_ev.reset_index()
}

for filename, table in tables.items():
    table.to_csv(table_dir / filename, index=False)

print(f"✅ Saved {len(tables)} analytical tables to {table_dir}")

✅ Saved 13 analytical tables to ..\outputs\tables


# 🏁 Notebook 03 — Final Analytical Deliverables

At the end of this notebook we have transformed the row-level EV dataset into:

### Aggregation Layer
- Manufacturer summaries
- State summaries
- County summaries
- City summaries
- EV-type summaries
- Model-year summaries
- Make × Model summaries
- Make × State summaries
- State × EV-type summaries

### Pivot Layer
- State × EV Type
- Make × EV Type
- Model Year × EV Type
- Range summaries
- MSRP summaries
- Multi-aggregation pivots
- Row-normalized percentage pivots

### Cross-Tab Layer
- Make × EV Type
- State × EV Type
- Make × CAFV
- State × CAFV
- Row percentage
- Column percentage
- Overall percentage

### Feature Engineering Layer
- Vehicle age
- Age groups
- Model-year groups
- Range groups
- MSRP groups
- Percentile features
- Z-score features
- Range/MSRP relationship features
- Manufacturer shares
- State shares
- EV-type shares
- CAFV shares
- Manufacturer/state/model ranks
- Binary business indicators

---

# 🔄 Next Notebook

The next stage should use this analytical dataset for:

```text
Notebook 01
Loading + Cleaning
        ↓
Notebook 02
Deep EDA
        ↓
Notebook 03
GroupBy + Pivot + Crosstab + Feature Engineering
        ↓
Notebook 04
Advanced Statistics + Hypothesis Testing
        ↓
Notebook 05
Advanced Visualization / Business Intelligence
        ↓
Notebook 06+
Machine Learning only where a valid target exists
```

## Final Rule

**Feature engineering here is analytical feature engineering, not ML preprocessing.**

We create useful business variables first. Model-specific transformations such as train/test splitting, encoding, scaling, leakage checks, and model pipelines should happen later in the Machine Learning notebooks.
